# Phase 8: Business Strategy & Insights
## Strategic Recommendations Based on Data Analysis
**Objective:** Generate actionable business insights and recommendations
**Output:** Business strategy insights for decision-making

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Libraries imported")

✓ Libraries imported


In [2]:
# Load cleaned data
df = pd.read_csv('Zomato_datasets/zomato_cleaned.csv')

# Create segments
df['price_segment'] = pd.qcut(df['avg_cost'], q=4, labels=['Budget', 'Mid-Range', 'Premium', 'Luxury'], duplicates='drop')
df['rating_category'] = pd.cut(df['rating'], bins=[0, 3, 3.5, 4, 4.5, 5], 
                                 labels=['Poor', 'Average', 'Good', 'Very Good', 'Excellent'], include_lowest=True)
df['primary_cuisine'] = df['cuisines'].apply(lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown')
df['service_level'] = df['has_online_order'] + df['has_table_booking']  # 0, 1, or 2

print(f"Dataset loaded: {df.shape}")

Dataset loaded: (7105, 16)


In [3]:
# 1. Success Factors - High Rated Restaurants Analysis
high_rated = df[df['rating'] >= 4.0]
low_rated = df[df['rating'] < 3.0]

success_factors = {
    'Online Order (High)': (high_rated['has_online_order'].mean() * 100),
    'Online Order (Low)': (low_rated['has_online_order'].mean() * 100),
    'Table Booking (High)': (high_rated['has_table_booking'].mean() * 100),
    'Table Booking (Low)': (low_rated['has_table_booking'].mean() * 100),
}

fig = go.Figure()

fig.add_trace(go.Bar(
    x=['Online Order', 'Table Booking'],
    y=[success_factors['Online Order (High)'], success_factors['Table Booking (High)']],
    name='High Rated (4.0+)',
    marker_color='#2ecc71'
))

fig.add_trace(go.Bar(
    x=['Online Order', 'Table Booking'],
    y=[success_factors['Online Order (Low)'], success_factors['Table Booking (Low)']],
    name='Low Rated (<3.0)',
    marker_color='#e74c3c'
))

fig.update_layout(
    title='Service Features: High-Rated vs Low-Rated Restaurants',
    yaxis_title='Percentage (%)',
    height=500,
    barmode='group'
)

fig.show()

print("\nSUCCESS FACTOR ANALYSIS:")
print("="*60)
print(f"Online Order availability in High-Rated (4.0+): {success_factors['Online Order (High)']:.1f}%")
print(f"Online Order availability in Low-Rated (<3.0): {success_factors['Online Order (Low)']:.1f}%")
print(f"Difference: {success_factors['Online Order (High)'] - success_factors['Online Order (Low)']:.1f}%")
print()
print(f"Table Booking availability in High-Rated (4.0+): {success_factors['Table Booking (High)']:.1f}%")
print(f"Table Booking availability in Low-Rated (<3.0): {success_factors['Table Booking (Low)']:.1f}%")
print(f"Difference: {success_factors['Table Booking (High)'] - success_factors['Table Booking (Low)']:.1f}%")


SUCCESS FACTOR ANALYSIS:
Online Order availability in High-Rated (4.0+): 56.3%
Online Order availability in Low-Rated (<3.0): 46.5%
Difference: 9.9%

Table Booking availability in High-Rated (4.0+): 40.7%
Table Booking availability in Low-Rated (<3.0): 2.8%
Difference: 37.9%


In [4]:
# 2. Segment Comparison Matrix
segment_comparison = df.groupby('price_segment').agg({
    'rating': ['mean', 'median'],
    'restaurant_name': 'count',
    'num_ratings': 'mean',
    'has_online_order': 'mean',
    'has_table_booking': 'mean'
}).round(2)

print("\nPRICE SEGMENT COMPARISON:")
print("="*60)
print(segment_comparison)

fig = go.Figure()

segments = ['Budget', 'Mid-Range', 'Premium', 'Luxury']
segment_data = df.groupby('price_segment').agg({
    'rating': 'mean',
    'restaurant_name': 'count'
}).reset_index()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Avg Rating by Price Segment', 'Restaurant Count by Segment')
)

fig.add_trace(
    go.Bar(x=segment_data['price_segment'], y=segment_data['rating'], 
           marker_color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'], name='Rating'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=segment_data['price_segment'], y=segment_data['restaurant_name'],
           marker_color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'], name='Count'),
    row=1, col=2
)

fig.update_yaxes(title_text='Avg Rating', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=2)
fig.update_layout(height=500, title_text='Price Segment Analysis', showlegend=False)
fig.show()


PRICE SEGMENT COMPARISON:
              rating        restaurant_name num_ratings has_online_order  \
                mean median           count        mean             mean   
price_segment                                                              
Budget          3.40    3.4            2708       55.20             0.49   
Mid-Range       3.45    3.5            1254       81.58             0.56   
Premium         3.47    3.5            1453      137.09             0.63   
Luxury          3.79    3.9            1690      527.41             0.46   

              has_table_booking  
                           mean  
price_segment                    
Budget                     0.00  
Mid-Range                  0.00  
Premium                    0.03  
Luxury                     0.41  


In [5]:
# 3. Opportunity Gaps Analysis
# Identify underserved areas
area_analysis = df.groupby('area').agg({
    'restaurant_name': 'count',
    'rating': 'mean',
    'has_online_order': 'mean'
}).reset_index()

area_analysis.columns = ['area', 'restaurant_count', 'avg_rating', 'online_order_pct']

# Identify high-demand but low-supply areas (high rating but few restaurants)
high_rating_areas = area_analysis[area_analysis['avg_rating'] >= 3.8].sort_values('restaurant_count')
underserved_areas = high_rating_areas[high_rating_areas['restaurant_count'] <= 10]

print("\nOPPORTUNITY GAPS - High Quality Areas with Low Competition:")
print("="*70)
print(underserved_areas.head(10).to_string())

fig = px.scatter(
    area_analysis,
    x='restaurant_count',
    y='avg_rating',
    size='online_order_pct',
    hover_name='area',
    title='Opportunity Gap Analysis (bubble size = online order availability %)',
    labels={'restaurant_count': 'Number of Restaurants', 'avg_rating': 'Average Rating'},
    color='avg_rating',
    color_continuous_scale='RdYlGn'
)

fig.update_layout(height=600)
fig.show()


OPPORTUNITY GAPS - High Quality Areas with Low Competition:
Empty DataFrame
Columns: [area, restaurant_count, avg_rating, online_order_pct]
Index: []


In [6]:
# 4. Market Gaps - Underserved Cuisines
cuisine_analysis = df.groupby('primary_cuisine').agg({
    'restaurant_name': 'count',
    'rating': 'mean',
    'avg_cost': 'mean'
}).reset_index()

cuisine_analysis.columns = ['cuisine', 'count', 'avg_rating', 'avg_cost']
cuisine_analysis = cuisine_analysis[cuisine_analysis['count'] >= 5]  # At least 5 restaurants

# High rating but few restaurants
high_demand_cuisines = cuisine_analysis[
    (cuisine_analysis['avg_rating'] >= 3.8) & 
    (cuisine_analysis['count'] <= cuisine_analysis['count'].quantile(0.25))
].sort_values('avg_rating', ascending=False)

print("\nMARKET GAPS - High-Demand Cuisines with Low Supply:")
print("="*70)
print(high_demand_cuisines.head(10).to_string())

fig = px.scatter(
    cuisine_analysis,
    x='count',
    y='avg_rating',
    size='avg_cost',
    hover_name='cuisine',
    title='Cuisine Market Gaps (bubble size = avg price)',
    labels={'count': 'Number of Restaurants', 'avg_rating': 'Average Rating'},
    color='avg_rating',
    color_continuous_scale='RdYlGn'
)

fig.update_layout(height=600)
fig.show()


MARKET GAPS - High-Demand Cuisines with Low Supply:
          cuisine  count  avg_rating     avg_cost
84     Vietnamese      5    4.260000  1400.000000
54  Modern Indian     13    4.246154  1526.923077
44         Korean      7    4.142857  1171.428571
31           Goan      8    3.900000   993.750000
65     Rajasthani     13    3.823077   576.923077


# 5. Strategic Recommendations Summary
print("\n" + "="*70)
print("BUSINESS STRATEGY RECOMMENDATIONS")
print("="*70)

print("\n1. SERVICE ADOPTION STRATEGY:")
print(f"   - {success_factors['Online Order (High)'] - success_factors['Online Order (Low)']:.1f}% more high-rated restaurants offer online orders")
print(f"   - Recommendation: Prioritize online ordering platform integration")

print("\n2. MARKET CONCENTRATION:")
top_10_areas = df['area'].value_counts().head(10).sum()
print(f"   - Top 10 areas contain {top_10_areas / len(df) * 100:.1f}% of all restaurants")
print(f"   - Recommendation: Focus expansion in underserved high-quality areas")

print("\n3. PRICE SEGMENT STRATEGY:")
best_segment = df.groupby('price_segment')['rating'].mean().idxmax()
best_rating = df.groupby('price_segment')['rating'].mean().max()
print(f"   - Best performing segment: {best_segment} (Avg Rating: {best_rating:.2f})")
print(f"   - Recommendation: Focus on {best_segment} segment for quality positioning")

print("\n4. CUISINE POSITIONING:")
top_3_cuisines = df['primary_cuisine'].value_counts().head(3)
print(f"   - Top 3 cuisines: {', '.join(top_3_cuisines.index)}")
print(f"   - Recommendation: Consider specialty cuisines for differentiation")

print("\n5. ENGAGEMENT INSIGHTS:")
online_avg = df[df['has_online_order'] == 1]['num_ratings'].mean()
no_online_avg = df[df['has_online_order'] == 0]['num_ratings'].mean()
print(f"   - Online order restaurants receive {online_avg:.0f} ratings on average")
print(f"   - Non-online restaurants receive {no_online_avg:.0f} ratings on average")
print(f"   - Multiplier: {online_avg / no_online_avg:.2f}x more engagement with online orders")

print("\n" + "="*70)
print("✓ Business Strategy Analysis Complete!")
print("="*70)